# Batched Execution

When you have many small tasks, each SLURM job has overhead (scheduling, startup). Batching
groups multiple items into a single SLURM job to reduce this overhead.

You'll learn how to:
- Use `units_per_worker` to batch items per SLURM job
- Handle remainders when items don't divide evenly
- Understand per-item failure isolation within batches

## Setup

In [ ]:
from prefect import flow, task
from prefect_submitit import SlurmTaskRunner

## Define Tasks

In [ ]:
@task
def identity(x):
    """Simple pass-through task."""
    return x


@task
def conditional_fail(x: int, fail_on: int = -1) -> int:
    """Returns x * 10, but raises ValueError if x == fail_on."""
    if x == fail_on:
        raise ValueError(f"intentional failure on {x}")
    return x * 10

## Basic Batching

With `units_per_worker=5`, 10 items are grouped into 2 SLURM jobs (5 items each).
Each future is a `SlurmBatchedItemFuture` that extracts its result from the batch.

In [ ]:
runner = SlurmTaskRunner(execution_mode="local", units_per_worker=5)


@flow(task_runner=runner)
def batched_flow():
    futures = identity.map(x=list(range(10)))
    results = [f.result() for f in futures]

    print(f"Results: {results}")
    print(f"Number of items: {len(futures)}")

    # Show the underlying SLURM jobs
    slurm_jobs = {f.slurm_job_future.slurm_job_id for f in futures}
    print(f"Number of SLURM jobs: {len(slurm_jobs)}")

    return results


results = batched_flow()
assert results == list(range(10))

## Remainder Handling

When items don't divide evenly, the last batch gets the remainder.
7 items with `units_per_worker=3` produces batches of [3, 3, 1].

In [ ]:
runner = SlurmTaskRunner(execution_mode="local", units_per_worker=3)


@flow(task_runner=runner)
def remainder_flow():
    futures = identity.map(x=list(range(7)))
    results = [f.result() for f in futures]

    print(f"Results: {results}")

    # Show batch grouping
    slurm_jobs = {f.slurm_job_future.slurm_job_id for f in futures}
    print(f"Number of SLURM jobs: {len(slurm_jobs)} (expected 3: batches of [3, 3, 1])")

    return results


results = remainder_flow()
assert results == list(range(7))

## Per-Item Failure Isolation

In batched execution, if one item within a batch fails, the other items in the same
batch still succeed. The failed item returns an error dict instead of raising an exception.

In [ ]:
runner = SlurmTaskRunner(execution_mode="local", units_per_worker=5)


@flow(task_runner=runner)
def failure_isolation_flow():
    # Item at index 2 will fail, others succeed
    futures = conditional_fail.map(x=[0, 1, 2, 3, 4], fail_on=2)
    results = [f.result() for f in futures]

    for i, r in enumerate(results):
        if isinstance(r, dict) and not r.get("success", True):
            print(f"Item {i}: FAILED - {r['error']}")
        else:
            print(f"Item {i}: {r}")

    return results


results = failure_isolation_flow()

# Successful items return x * 10
assert results[0] == 0
assert results[1] == 10
assert results[3] == 30
assert results[4] == 40

# Failed item returns an error dict
assert isinstance(results[2], dict)
assert results[2]["success"] is False
print(f"\nError details: {results[2]}")